In [ ]:
import math

# 1. 定义已知变量
delta_r_i = 6.5       # 岩石初始值
delta_r_f = -4.0      # 岩石终值
delta_w_i = -14.0     # 水初始值
T_celsius = 500       # 摄氏度
T_k = T_celsius + 273.15 # 转换为开尔文温度
An = 0.3              # 钙长石摩尔分数

# 2. 计算同位素分馏系数 Delta (1000 ln alpha)
# 公式: (2.91 - 0.76An)(10^6 T^-2) + (-3.41 - 0.14An)
term1 = (2.91 - 0.76 * An) * (1e6 / (T_k ** 2))
term2 = -3.41 - 0.14 * An
Delta = term1 + term2

print(f"在 {T_celsius}℃ 下，斜长石-水的氧同位素分馏系数 (Delta) 约为: {Delta:.4f} ‰")

# 3. 计算封闭体系下的 W/R 比值
# 公式: W/R = (delta_r_f - delta_r_i) / (delta_w_i - (delta_r_f - Delta))
numerator_closed = delta_r_f - delta_r_i
denominator_closed = delta_w_i - (delta_r_f - Delta)
WR_closed = numerator_closed / denominator_closed

print(f"【封闭体系】计算得出的 W/R 比值为: {WR_closed:.4f}")

# 4. 计算开放体系下的 W/R 比值
# 公式: W/R = ln[ (delta_r_i - delta_w_i - Delta) / (delta_r_f - delta_w_i - Delta) ]
numerator_open = delta_r_i - delta_w_i - Delta
denominator_open = delta_r_f - delta_w_i - Delta

# 确保对数里的值为正数
if numerator_open / denominator_open > 0:
    WR_open = math.log(numerator_open / denominator_open)
    print(f"【开放体系】计算得出的 W/R 比值为: {WR_open:.4f}")
else:
    print("错误：对数内部计算结果为负数，请检查参数设置。")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 已知条件
# 初始玄武岩 δ18O = 6‰，海水 δ18O = 0‰，闭合体系 W/R = 10
delta_r_i = 6.0
delta_w_i = 0.0
WR = 10.0

# 温度范围（℃）
T_celsius = np.linspace(0, 800, 401)
T_k = T_celsius + 273.15

# 玄武岩-海水分馏系数 Δ（1000 ln α）
# Δ = 3.42 × 10^6 T^-2 - 3.90 × 10^3 T^-1
Delta = 3.42e6 / (T_k ** 2) - 3.90e3 / T_k
alpha = np.exp(Delta / 1000.0)

# 闭合体系质量守恒：先用同位素比值计算，再转回 δ 表示
r_r_i = delta_r_i / 1000.0 + 1.0
r_w_i = delta_w_i / 1000.0 + 1.0
r_r_f = alpha * (r_r_i + WR * r_w_i) / (WR + alpha)
delta_r_f = (r_r_f - 1.0) * 1000.0

# 输出几个参考温度点
for temp in [100, 300, 500, 700]:
    idx = np.argmin(np.abs(T_celsius - temp))
    print(f"{T_celsius[idx]:.0f}℃ 时，蚀变玄武岩的 δ18O = {delta_r_f[idx]:.3f} ‰")

# 绘制曲线
plt.figure(figsize=(8, 5))
plt.plot(T_celsius, delta_r_f, color="darkred", linewidth=2)
plt.xlabel("Alteration temperature (°C)")
plt.ylabel("Altered basalt δ18O (‰)")
plt.title("Closed system: W/R = 10, basalt δ18O vs. temperature")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Initial normalized isotope ratios
Rr_i = 1.0 + 6.0 / 1000.0
Rw_i = 1.0 + 0.0 / 1000.0

# W/R range: 0.01 to 100 (log-spaced)
wr = np.logspace(-2, 2, 500)


def delta_final(T_celsius, wr_array):
    """Compute altered basalt δ18O at a fixed temperature for closed system."""
    T_k = T_celsius + 273.15
    Delta = 3.42e6 * (T_k ** -2) - 3.90e3 * (T_k ** -1)
    alpha = np.exp(Delta / 1000.0)

    Rr_f = alpha * (Rr_i + wr_array * Rw_i) / (wr_array + alpha)
    delta_r_f = (Rr_f - 1.0) * 1000.0
    return delta_r_f


# Two temperatures
T1 = 2.0
T2 = 400.0

curve_T1 = delta_final(T1, wr)
curve_T2 = delta_final(T2, wr)

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
ax.semilogx(wr, curve_T1, color="blue", linewidth=2, label="2°C (low T)")
ax.semilogx(wr, curve_T2, color="red", linewidth=2, label="400°C (high T)")

ax.set_xlabel("W/R ratio (Molar)")
ax.set_ylabel("Altered Basalt δ18O (‰)")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

plt.tight_layout()
plt.show()